# 第 3 章 · Agent 配置、State 与结构化输出

> **你已会**：第 1、2 章的 Agent、Runner、Session 与事件。
>
> **读完后你能**：区分历史与 State，解释状态提交，再用 instruction、output_key 与本地校验完成一个小任务。
>
> **边界**：本章先讲 Session 内的键值；共享作用域、持久化与 Memory 在第 6 章。工具与多 Agent 在第 4、5 章。

主线先运行下面的离线实验，再读参数；每段只新增一个机制。在线诗词、生成参数与真实 JSON 输出用于观察模型，明确需要密钥。

## 0. 历史里有一句话，State 就会自动多一个键吗？

先分清 Session 里的两个容器：

| 容器 | 是什么 | 谁写 | 模型能看到吗 |
|---|---|---|---|
| 历史 `session.events` | 按时间追加的 `Event` 列表：用户消息、模型回复、工具调用与返回 | 框架在每轮交互中追加 | 能。历史会被组装成模型上下文 |
| 状态 `session.state` | 当前会话的键值快照 `dict` | 程序：创建会话时的初始值、`output_key`、工具、回调或事件增量 | 不能。除非用 `{key}` 插值，或工具主动把它读出来 |

一条事件除了 `content`（这一轮说了什么），还带 `actions`。状态变更就走 `actions.state_delta`：

```mermaid
flowchart LR
    U["用户消息"] --> E1["Event 1"]
    E1 --> E2["Event 2<br/>actions.state_delta = {'draft': 'v1'}"]
    E2 --> H["session.events<br/>只追加，不修改"]
    E2 -. "append_event 时合并 delta" .-> S["session.state<br/>{'draft': 'v1'}"]
```

这条链是单向、显式的：

> 事件（事实记录）→ `state_delta`（变更意图）→ State（当前快照）

历史里出现一句话，不会自动长出 State 键。运行期的 State 更新只有一条路径：携带 `state_delta` 的事件被提交；`create_session(state=...)` 是创建时的初始写入，不属于这条运行期链路。反过来，State 变化也不会自动改写历史。

本章只使用会话级键（无前缀）。`user:` / `app:` / `temp:` 作用域、持久化后端和 Memory 在第 6 章展开。

**先预测**：初始化 `state={"draft": "v0"}` 后，直接改取回的 Session，再重新读会话，会看到 v1 吗？

In [1]:
from google.adk.events import Event, EventActions
from google.adk.sessions import InMemorySessionService
from google.genai import types

state_service = InMemorySessionService()
state_session = await state_service.create_session(
    app_name="state_intro", user_id="student", session_id="one",
    state={"draft": "v0"},  # API 参数名是 state，不是 initial_state
)
state_session.state["draft"] = "只改了取回的副本"  # ① get_session() 返回的是副本
fresh = await state_service.get_session(app_name="state_intro", user_id="student", session_id="one")
assert fresh.state["draft"] == "v0"                # ② 副本上的修改没有提交

await state_service.append_event(                   # ③ 只有内容、没有 state_delta：历史变长，State 不变
    fresh,
    Event(author="user", content=types.Content(role="user", parts=[types.Part(text="这句只是聊天")])),
)
still = await state_service.get_session(app_name="state_intro", user_id="student", session_id="one")
assert still.state["draft"] == "v0"
assert "这句只是聊天" in still.events[-1].content.parts[0].text

await state_service.append_event(                   # ④ 事件携带 state_delta 才能提交
    still,
    Event(author="lesson", actions=EventActions(state_delta={"draft": "v1"}),
          content=types.Content(role="user", parts=[types.Part(text="保存草稿")])),
)
saved = await state_service.get_session(app_name="state_intro", user_id="student", session_id="one")
assert saved.state["draft"] == "v1"                # ⑤ State 已更新
assert saved.events[-1].actions.state_delta == {"draft": "v1"}  # ⑥ 事件仍保留增量
assert "保存草稿" in saved.events[-1].content.parts[0].text
print("历史记录：", saved.events[-1].content.parts[0].text)
print("提交后的 State：", saved.state)

历史记录： 保存草稿
提交后的 State： {'draft': 'v1'}


**证据**：② 直接改副本无效；③ 只有内容的事件只让历史变长，State 不变；④⑤ 事件携带 `state_delta` 后，重新读取才看到 `draft=v1`；⑥ 变更仍留在历史里。事件是提交动作，State 是提交后的快照。

工具中的 `tool_context.state[key] = value` 走的是同一机制：框架把改动收进该次工具调用的 `state_delta`，随函数响应事件提交。嵌套字典应先复制、修改，再重新赋给顶层 key；第 4 章会验证它。先按这个模型读下文的插值与 output_key，不必先背全部参数。

## 1. Agent 的参数全景

`Agent` 是 `LlmAgent` 的别名。参数很多，但常用的就那几个。先看全貌：

| 参数 | 作用 | 什么时候会用到 |
|---|---|---|
| `name` | 智能体唯一标识 | 永远 |
| `model` | 底层模型 | 永远 |
| `instruction` | 人设、规则、输出格式 | 永远 |
| `description` | 一句话能力描述，供上级 Agent 路由 | 第 5 章 |
| `tools` | 可调用的工具 | 第 4 章 |
| `sub_agents` | 下级 Agent | 第 5 章 |
| `output_key` | 把最终回复写入 Session 状态 | 步骤间传递结果 |
| `output_schema` | 要求结构化输出 | 需要 JSON 时（见 6 的 DeepSeek 限制） |
| `generate_content_config` | 温度、最大输出长度等 | 需要控制风格或成本时 |
| `before/after_*_callback` | 生命周期钩子 | 第 7 章 |
| `mode` | 对话 / 任务 / 单轮 | 第 8 章做工作流节点时 |
| `include_contents` | 是否把历史带进请求 | 无状态单轮任务可设 `none` 省 token |
| `planner` / `code_executor` | 规划器 / 代码执行器 | 进阶，本教程不展开 |

> 📌 **架构上的一个事实**：`Agent.model` 接受的是 `BaseLlm` 接口的实例（例如 `LiteLlm`），而不是某个具体厂商的客户端。ADK 因此不把运行时绑死在单一模型上。本教程只使用 `LiteLlm` + DeepSeek 这一条路径。

<details>
<summary><b>深入：<code>mode</code> 的默认值为什么"看场合"</b></summary>

`mode` 默认为 `None`，实际行为随使用方式而定：作为子 Agent 时默认 `chat`，作为工作流节点时默认 `single_turn`。

两者的真正区别不是"参不参与对话"，而是**输入从哪来**：

| 模式 | 输入来源 |
|---|---|
| `chat` | 会话历史（session events） |
| `single_turn` | 上游节点直接传入的 `node_input` |

所以把 `chat` 模式的 Agent 接在别的节点后面时，ADK 在构造工作流图阶段就会拒绝——它拿不到上游数据。第 8 章会真正用上这个区别。

</details>

---

## 2. DeepSeek 是怎么接进来的

ADK 原生字符串模型名走 Google 的模型路径。本教程统一传入一个 `LiteLlm` 对象：

```python
from google.adk.models.lite_llm import LiteLlm

model = LiteLlm(model="deepseek/deepseek-chat")
```

这里只有一条规则需要记：**`provider/model`**。

- `provider` 是 `deepseek`，LiteLLM 据此把请求路由到 DeepSeek；
- 同时它会读取环境变量 `DEEPSEEK_API_KEY`；
- `model` 部分是具体模型名 `deepseek-chat`。

这条路径不需要 `GOOGLE_API_KEY`，也不需要任何 Google Cloud 凭证。

> ⚠️ **唯一需要额外注意的地方**：DeepSeek 的 API 与 Gemini 并不完全等价。本教程锁定的 LiteLLM 线路使用 `response_format={"type": "json_object"}` 做结构化输出；`json_schema` 是否可用取决于供应商与适配器版本，换版本前先验证。这一点直接决定了 6 的写法。

先用第 1 章的方式重建运行环境：

In [2]:
import os
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP, USER = "adk_ch03", "student"

async def run_once(agent, query, session_id="demo", verbose=False):
    """通用一次性问答：新建 Runner+Session，返回最终文本（可选打印事件流）。"""
    ss = InMemorySessionService()
    await ss.create_session(app_name=APP, user_id=USER, session_id=session_id)
    runner = Runner(agent=agent, app_name=APP, session_service=ss)
    msg = types.Content(role="user", parts=[types.Part(text=query)])
    final = ""
    async for ev in runner.run_async(user_id=USER, session_id=session_id, new_message=msg):
        # verbose(啰嗦) 将打印所有event
        if verbose and ev.content and ev.content.parts:
            for p in ev.content.parts:
                if p.text: print(f"  [event:{ev.author}] {p.text[:80]}")
        if ev.is_final_response() and ev.content and ev.content.parts:
            final = ev.content.parts[0].text
    # 顺便返回 session 便于查看状态
    session = await ss.get_session(app_name=APP, user_id=USER, session_id=session_id)
    return final, session

print("✅ 运行环境就绪")


✅ 运行环境就绪


---
## 3. `instruction`：不止是一段静态文字

### 3.1 基础用法：角色 + 规则 + 输出格式

一段能用的 `instruction` 通常有三块：

1. **角色**：你是谁；
2. **行为准则**：遇到什么情况怎么做；
3. **输出要求**：格式、长度、语气。

先猜：**如果只写"你是一位文学老师"，不写任何格式要求，回答会长什么样？** 运行后对照你的猜测。


In [3]:
poet = Agent(
    name="poet",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="""你是一位精通中国古典诗词的文学老师。
行为准则：
1. 赏析诗词时先给白话翻译，再给艺术点评；
2. 点评不超过 100 字；
3. 结尾推荐一首风格相近的诗。
""",
    description="中国古典诗词赏析老师",
)

text, _ = await run_once(poet, "赏析一下「大漠孤烟直，长河落日圆」。")
print(text)


**白话翻译**：浩瀚沙漠中一缕烽烟笔直上升，黄河尽头一轮落日浑圆低垂。

**艺术点评**：十字写尽塞外苍茫。以“直”状孤烟之劲拔，以“圆”摹落日之温润，线条与几何感极强；大漠、长河铺展平面，孤烟、落日构成纵与横的对照，画面开阔而静穆，王国维誉为“千古壮观”。

**推荐**：王维《使至塞上》全篇可对读，另荐王之涣《登鹳雀楼》，同具雄浑开阔之境。


### 3.2 状态插值：让 instruction "活"起来

`instruction` 里的 `{key}` 会在**发送给模型之前**，从 Session 状态里取值并替换。

```python
instruction="你正在和 {username} 对话，TA 的母语是 {native_lang}。"
```

状态值在创建会话时通过 `create_session(state=...)` 写入。于是同一个 Agent 定义，可以对不同用户呈现不同人设。

**先猜：如果状态里没有 `username`，这段 instruction 会原样发给模型，还是报错？** 先用下面的离线实验检查传给模型的 instruction，再看真实模型示例。

In [4]:
from google.adk.models.base_llm import BaseLlm
from google.adk.models import LlmResponse

class NeverCalledModel(BaseLlm):
    model: str = "offline_interpolation"
    async def generate_content_async(self, llm_request, stream=False):
        raise AssertionError("实验应由 before_model_callback 截获，不调用供应商")
        yield  # 保持异步生成器接口

async def interpolation_case(template, state):
    seen = []
    def inspect_instruction(callback_context, llm_request):
        seen.append(llm_request.config.system_instruction)
        return LlmResponse(content=types.Content(role="model", parts=[types.Part(text="已检查")]))
    local = InMemorySessionService()
    await local.create_session(app_name=APP, user_id=USER, session_id="interpolation", state=state)
    agent = Agent(name="interpolation", model=NeverCalledModel(),
                  instruction=template, before_model_callback=inspect_instruction)
    runner = Runner(agent=agent, app_name=APP, session_service=local)
    error = None
    try:
        async for _ in runner.run_async(user_id=USER, session_id="interpolation",
                new_message=types.Content(role="user", parts=[types.Part(text="开始")])):
            pass
    except KeyError as exc:
        error = exc
    return seen, error

required, missing_error = await interpolation_case("姓名={username}", {})
optional, optional_error = await interpolation_case("姓名={username?}", {})
present, present_error = await interpolation_case("姓名={username}", {"username": "小林"})
assert missing_error is not None and required == []
assert optional_error is None and "姓名=" in optional[0]
assert present_error is None and "姓名=小林" in present[0]
print({"必填缺失": type(missing_error).__name__, "可选缺失": optional[0], "有值": present[0]})

Node execution failed with exception
Traceback (most recent call last):
  File "D:\Python\Lib\site-packages\google\adk\workflow\_node_runner.py", line 136, in run
    await self._execute_node(ctx, node_input)
  File "D:\Python\Lib\site-packages\google\adk\workflow\_node_runner.py", line 274, in _execute_node
    await self._run_node_loop(ctx, node_input)
  File "D:\Python\Lib\site-packages\google\adk\workflow\_node_runner.py", line 288, in _run_node_loop
    async for event in agen:
  File "D:\Python\Lib\site-packages\google\adk\workflow\_base_node.py", line 166, in run
    async for item in agen:
  File "D:\Python\Lib\site-packages\google\adk\agents\llm_agent.py", line 599, in _run_impl
    async for event in agen:
  File "D:\Python\Lib\site-packages\google\adk\workflow\_llm_agent_wrapper.py", line 370, in run_llm_agent_as_node
    async for event in run_iter:
  File "D:\Python\Lib\site-packages\google\adk\agents\base_agent.py", line 306, in run_async
    async for event in agen:
  Fi

{'必填缺失': 'KeyError', '可选缺失': '姓名=\n\nYou are an agent. Your internal name is "interpolation".', '有值': '姓名=小林\n\nYou are an agent. Your internal name is "interpolation".'}


**证据**：必填值缺失时，框架在进入模型回调之前报错；`{username?}` 允许缺值，替换为空文本。**迁移**：可选值不等于业务默认值；需要“普通用户”等默认值时，请由应用显式写入 State。

> 输出里的长 traceback 是 ADK 记录预期 `KeyError` 的日志，不是实验失败；最后一行 `{'必填缺失': 'KeyError', ...}` 才是实验结果。

In [5]:
greeter = Agent(
    name="greeter",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="你正在和 {username} 对话，TA 的母语是 {native_lang}。请用对方的母语打招呼，并自我介绍。",
    description="多语言问候助手",
)

# 使用 create_session(state=...) 预置会话状态
ss = InMemorySessionService()

await ss.create_session(app_name=APP, user_id=USER, session_id="i18n",
                        state={"username": "小明", "native_lang": "中文"}) # ← 关键: 这里使用 state:dict的方式传入参数
runner = Runner(agent=greeter, app_name=APP, session_service=ss)
msg = types.Content(role="user", parts=[types.Part(text="开始吧")])
async for ev in runner.run_async(user_id=USER, session_id="i18n", new_message=msg):
    if ev.is_final_response():
        print(ev.content.parts[0].text)


你好，小明！我是多语言问候助手，很高兴认识你。


> ⚠️ **三个容易踩的点**：
>
> 1. `{key}` 是**指令注入**，替换发生在调用模型之前。改了状态之后，下一轮才会生效；
> 2. **键不存在时会直接抛 `KeyError: Context variable not found`**，不是原样保留，也不会自动补默认值；
> 3. 确实可能缺失的字段，写成 **`{key?}`**（问号表示可选）：缺失或值为 `None` 时替换成空字符串，不会中断运行。
>
> 所以"可选信息"要在 instruction 里显式用 `?` 标出来，并约定空值时的行为。

---

## 4. `output_key`：把回复存进 Session 状态

默认情况下，Agent 的回复只出现在事件流里。

加上 `output_key` 后，最终文本会**自动写入 `session.state[output_key]`**。后续的 Agent、工具或你自己的代码都能直接读取。

这是 Agent 共享状态的一种方式。第 5 章当前 Workflow 主线采用返回值传数据；迁移深读保留 output_key 接力示例。

**先猜：不加 `output_key`，上一步的回复还能从 Session 状态里直接取到吗？**

In [6]:
summarizer = Agent(
    name="summarizer",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="把用户给的文本总结成一句话。",
    description="一句话总结器",
    output_key="summary",   # ← 关键：最终回复存入 state["summary"]
)

text, session = await run_once(summarizer, "人工智能正在深刻改变软件开发的方式，从代码补全到自动测试，再到今天的智能体协同编程，开发者的角色正从『写代码的人』转变为『指挥智能体的人』。")
print("Agent 回复：", text)
print("─" * 50)
print("Session 状态中已归档：", session.state.get("summary"))


Agent 回复： 人工智能正推动开发者从写代码转向指挥智能体协同编程。
──────────────────────────────────────────────────
Session 状态中已归档： 人工智能正推动开发者从写代码转向指挥智能体协同编程。


---

## 5. `generate_content_config`：控制随机性与长度

温度（`temperature`）控制随机性：

- `0` 附近：更稳定、更确定，适合事实型任务；
- `1` 以上：更发散，适合创意型任务。

ADK 用 `GenerateContentConfig` 表达这些生成参数：

```python
generate_content_config=GenerateContentConfig(
    temperature=1.2,
    max_output_tokens=500,
)
```

> 📌 **不要只看单次输出下结论。** 同一个提示词至少跑三次，才能看出"稳定"还是"碰巧"。第 1 题练习就是让你用数据回答这个问题。

In [7]:
from google.genai.types import GenerateContentConfig

creative = Agent(
    name="creative",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="给产品起 3 个中文名，每个附一句 slogan。",
    description="品牌创意命名师",
    generate_content_config=GenerateContentConfig(temperature=1.2, max_output_tokens=500),
)

text, _ = await run_once(creative, "产品：一款帮助程序员管理久坐健康的 App。")
print(text)


1. **久安** —— 坐下写代码，健康有人守。
2. **码上起** —— 码到十分钟，起身一分钟。
3. **坐伴** —— 久坐不孤单，健康常相伴。


---

## 6. 结构化输出：在 DeepSeek 上怎么做

业务系统要的往往不是"一段话"，而是**一个能入库的 JSON**。

ADK 提供了 `output_schema` 参数。它可能使用原生 schema 或框架辅助工具，具体路径取决于 ADK、LiteLLM 与模型版本。本课不把单个适配器的历史报错当作所有模型的限制。

本仓库锁定的 DeepSeek 线路采用 JSON mode 加本地校验；它与供应商原生 schema 约束不是同一种保证：

| 环节 | 用什么保证 |
|---|---|
| 输出是 JSON | `response_format={"type": "json_object"}` |
| 字段齐全、类型正确 | Pydantic 模型校验 |
| 业务规则（长度、取值范围、关键词） | 自己写检查函数 |

> 📌 **结构化输出 = 格式约束 + 字段校验 + 业务检查。** 三段缺一不可。解析与校验仍必须执行；没有这些检查，合法 JSON 也可能包含错误字段或事实。

下面用 Pydantic 定义一个书评结构，让模型按契约输出，再在本地校验：


In [8]:
import json
from pydantic import BaseModel, ConfigDict, Field, ValidationError

class BookReview(BaseModel):
    model_config = ConfigDict(extra="forbid")
    title: str = Field(min_length=1)
    rating: int = Field(ge=1, le=5, strict=True)
    keywords: list[str] = Field(min_length=3, max_length=3)
    one_line: str = Field(min_length=1)

def validate_review(text: str, expected_title: str) -> BookReview:
    raw = json.loads(text)                       # 格式层
    review = BookReview.model_validate(raw)       # 字段层
    if review.title != expected_title:            # 业务层
        raise ValueError("书名与输入任务不一致")
    return review

valid_review = {"title": "三体", "rating": 5, "keywords": ["科幻", "宇宙", "文明"], "one_line": "文明相遇的故事"}
assert validate_review(json.dumps(valid_review), "三体").rating == 5
for raw, error_type in [
    ("不是 JSON", json.JSONDecodeError),
    (json.dumps({**valid_review, "rating": 8}), ValidationError),
    (json.dumps({**valid_review, "title": "别的书"}), ValueError),
]:
    try:
        validate_review(raw, "三体")
    except error_type as exc:
        print("预期失败：", type(exc).__name__)
    else:
        raise AssertionError("应拒绝此反例")

预期失败： JSONDecodeError
预期失败： ValidationError
预期失败： ValueError


**迁移**：离线反例已确认三层边界。下面才启用真实 JSON mode，失败必须保留并诊断，不能剥掉错误后假装契约已满足。

In [9]:
reviewer_ds = Agent(
    name="reviewer_ds",
    model=LiteLlm(model="deepseek/deepseek-chat", response_format={"type": "json_object"}),
    instruction='只输出 JSON 书评：title、1到5整数 rating、恰好3项 keywords、one_line。不要用 Markdown 围栏。',
    description="书评结构化生成器",
    output_key="review_json",
)
text, session = await run_once(reviewer_ds, "请为《三体》生成简短书评，title 必须为三体。")
review = validate_review(text, "三体")
assert session.state["review_json"] == text
print(review.model_dump())

{'title': '三体', 'rating': 5, 'keywords': ['硬科幻', '宇宙社会学', '文明史诗'], 'one_line': '一部以黑暗森林法则震撼人心的中国科幻巅峰之作。'}


**能力边界**：`output_schema + tools` 不是 ADK 全局硬禁组合。框架版本、模型和适配器决定实际路径，可能涉及结构化输出辅助工具；必须按目标组合验证。拆成“取数→格式化”是便于测试和隔离职责的设计选择，并非所有应用的强制要求。

<details>
<summary><b>深入：与 LangChain 的结构化输出对照</b></summary>

LangChain 的结构化输出也可能选择工具调用通道；可用性仍需按具体模型与适配器验证。同样是结构化输出，两个框架的底层通道不同，兼容性和失败模式也不同。

值得记住的不是"谁更好"，而是：**任何框架的结构化输出，最终都要落到"模型支持哪种通道 + 你如何校验"这两个问题上。**

</details>

---

## 7. `description`：为多智能体埋下伏笔

`description` 在 Agent 自己运行时几乎不起作用。它的舞台在第 5 章：父 Agent 决定"把任务转交给哪个子 Agent"时，读的就是每个子 Agent 的 `description`。

因此它应该像一份**给调度系统看的岗位简介**：写清擅长什么、什么情况下该被选中。

```python
# ❌ 说了等于没说
description="一个很有用的助手"

# ✅ 说清了边界和触发条件
description="处理订单退款相关的请求，包括查询退款进度和发起退款申请"
```

> ✍️ 判断标准：把描述给另一个人看，他能否判断"这个问题该不该找它"。

---

<details>
<summary><b>深入：与 LangChain / LangGraph 对照</b></summary>

| ADK 概念 | LangChain / LangGraph 对应 | 差异 |
|---|---|---|
| `Agent(instruction=...)` | `create_agent(model, system_prompt=...)` | 几乎一一对应 |
| `LiteLlm(model="deepseek/...")` | `ChatOpenAI(base_url=...)` 等 | 接法不同，目标一致 |
| `output_key` 写入 state | 节点函数返回 `{"key": value}` | ADK 声明式一行；LangGraph 显式但灵活 |
| `generate_content_config` | 模型构造参数 | ADK 与 Agent 绑定，LangChain 与模型绑定 |
| instruction `{state}` 插值 | `ChatPromptTemplate` 变量 | ADK 直接读 Session state，少一层模板 |

</details>

---

## 📌 用五句话复述本章

1. **参数不用背**，先知道 `instruction` / `output_key` / `description` 各自负责什么。
2. **`instruction` 的 `{key}` 是调用模型前的状态替换**，缺值时不会自动补默认值。
3. **`output_key` 把最终回复写进 Session 状态**，是多 Agent 接力的基础。
4. **本课的 DeepSeek 示例使用 JSON mode**，再做 Pydantic 字段校验与业务检查；换模型或适配器后重新验证能力。
5. **`description` 是写给调度者看的**，要写清边界和触发条件。

> ➡️ 下一章：[04-工具系统](04-工具系统.ipynb) —— 让 Agent 从"会说"进化到"会做"。


---

## 🧪 本章练习

### 1. 参数实验：让差异可观察（基础）

固定同一提示词，只改 `generate_content_config`。

- [ ] 至少设置两组不同参数（例如低温 + 短输出、高温 + 长输出）
- [ ] 每组运行三次
- [ ] 用表格记录输出稳定性、格式遵循度、长度和主观质量
- [ ] 说明哪组适合事实型助手、哪组适合创意型助手

### 2. 动态 instruction 与状态接力（进阶）

做一个客服分诊 Agent。

- [ ] 从 Session State 读取用户等级、语言、产品类型
- [ ] 动态改变语气与处理规则
- [ ] 用 `output_key` 保存标准化分诊结论
- [ ] 覆盖"缺少状态值 / 普通用户 / VIP 用户"三种情况
- [ ] 确认状态中保留了下一 Agent 可用的结果

### 3. 结构化输出的工程边界（进阶）

为"会议行动项"设计输出契约。

- [ ] Pydantic Schema 至少包含负责人、任务、截止日期、优先级、风险提示
- [ ] 用 `json_object` + instruction 契约生成结果，并在本地校验
- [ ] 比较单 Agent 结构化输出与“先取数、后格式化”的两阶段设计，说明模型能力与数据契约
- [ ] 写清两个阶段之间的数据契约

### 4. 编写可路由的岗位说明（工程）

为退款、物流、技术支持、兜底四个子 Agent 写 `name`、`description` 和关键 `instruction`。

- [ ] 至少构造八条容易混淆的用户请求
- [ ] 人工评审 description 是否足以支持正确路由
- [ ] 把失败样例整理成回归集，第 5 章直接复用

### 5. 把 instruction 变成可测试的契约（工程）

选一个本章的 Agent，为它写一组离线检查。

- [ ] 至少覆盖三种输入：正常、边界、应该拒绝的请求
- [ ] 用代码断言回复的结构和关键字段，而不是"看起来不错"
- [ ] 说明哪些检查是确定性的、哪些依赖模型输出，需要多跑几次
- [ ] 记录一次失败样例，并写出你会怎么改 instruction